In [2]:

import requests
import mlflow
from pipelines.data_preprocessing import (
    chunk_text,
    load_text,
    save_chunks_to_jsonl,
)
from pipelines.embeddings import compute_embeddings
from pipelines.rag_pipeline import (
    answer_query,
    build_rag_index,
    llm_answer_query,
    log_query,
    log_rag_version,
)


%load_ext autoreload
%autoreload 2


mlflow_uri = None

for uri in ["http://localhost:5000", "http://mlflow:5000"]:
    try:
        r = requests.get(f"{uri}/api/2.0/mlflow/experiments/search?max_results=1", timeout=2)
        if r.status_code == 200:
            mlflow_uri = uri 
            break
    except requests.RequestException:
        continue

if mlflow_uri is None:
    raise RuntimeError("MLflow сервер недоступен по обоим адресам")

mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("RAG_Experiment")

text_url = "https://blog.dzencode.com/ru/illyuziya-kachestva-vash-sayt-idealen-pozdravlyaem-vy-tolko-chto-sozhgli-byudzhet/"
article_text = load_text(text_url)

chunk_size =300
model_name="all-MiniLM-L6-v2"
chunks = chunk_text(article_text, chunk_size, semantic=False)

with mlflow.start_run(run_name="SentenceTransformers_v1"):
    
    chunks_with_embeddings=compute_embeddings(chunks, model_name)
    save_chunks_to_jsonl(chunks_with_embeddings)

    host = "localhost"
    port = 6333
    try:
        r = requests.get(f"http://{host}:{port}", timeout=2)
        r.raise_for_status()
    except Exception:
        host = "qdrant"
    
    index = build_rag_index(chunks_with_embeddings,host=host, port=port)
    mlflow.log_param("embedding_model", model_name)
    mlflow.log_param("chunk_size", chunk_size)

    query = "Что значит «Иллюзия качества»?"
    answer = answer_query(index, query)
    print(answer)


    %load_ext autoreload
    %autoreload 2

    query = "Что автор имеет в виду под 'иллюзией качества'?"
    llm_answer = llm_answer_query(query, answer)
    rag_file_path = "artifacts/rag_article.jsonl"
    print("Ответ LLM:", llm_answer)
    log_query(query ,llm_answer,"logs/query_log.json")
    log_rag_version(rag_file_path, "version_log.json")


    mlflow.log_artifact(rag_file_path)          # RAG-файл

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


KeyboardInterrupt: 

In [4]:
from pipelines.data_preprocessing import (
    chunk_text,
    load_text,
    save_chunks_to_jsonl,
)
%load_ext autoreload
%autoreload 2

text_url = "https://blog.dzencode.com/ru/illyuziya-kachestva-vash-sayt-idealen-pozdravlyaem-vy-tolko-chto-sozhgli-byudzhet/"
article_text = load_text(text_url)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


chank data

In [5]:
from pipelines.data_preprocessing import chunk_text, save_chunks_to_jsonl
%load_ext autoreload
%autoreload 2
chunks = chunk_text(article_text, chunk_size=512, semantic=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from pipelines.embeddings import compute_embeddings
from pipelines.rag_pipeline import build_rag_index
%load_ext autoreload
%autoreload 2
chunks_with_embeddings=compute_embeddings(chunks, model_name="all-MiniLM-L6-v2")
save_chunks_to_jsonl(chunks_with_embeddings)
index = build_rag_index(chunks_with_embeddings)

c:\dzen\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Batches: 100%|██████████| 2/2 [00:05<00:00,  2.59s/it]


Сгенерировано 37 эмбеддингов, размерность 384
✅ RAG-файл сохранён: artifacts/rag_article.jsonl
В Qdrant загружено 37 чанков в коллекцию 'dzencode_articles'


In [7]:
from pipelines.rag_pipeline import answer_query
%load_ext autoreload
%autoreload 2
query = "Что значит «Иллюзия качества»?"
answer = answer_query(index, query)
print(answer)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Практика: галерея блестящих провалов
Добро пожаловать на нашу эксклюзивную экскурсию по залу славы “Иллюзии качества”. Здесь собраны настоящие шедевры, которые стоили своим владельцам целое состояние. Билеты не нужны. Просто смотрите и узнавайте.
Экспонат №1: “Сайт-музей”
Диагноз: Синдром Awwwards. На совещаниях по проекту слово “красиво” звучит чаще, чем слово “конверсия”?
Утверждая дизайн, вы и ваша команда ориентируетесь на личный вкус ( “мне нравится” ), а не на данные о поведении пользователей?
Утверждая дизайн, вы и ваша команда ориентируетесь на личный вкус ( “мне нравится” ), а не на данные о поведении пользователей?
Главным результатом работы подрядчика вы считаете “сдачу макетов в срок” , а не “достижение бизнес-KPI после запуска”? P.S. Клуб анонимных перфекционистов
Кстати, о “памятниках”. Какой самый вопиющий пример “Иллюзии качества” вы встречали в своей жизни? Расскажите в комментариях

In [ ]:
from pipelines.rag_pipeline import llm_answer_query, log_query,log_rag_version

%load_ext autoreload
%autoreload 2

query = "Что автор имеет в виду под 'иллюзией качества'?"
llm_answer = llm_answer_query(query, answer)
rag_file_path = "artifacts/rag_article.jsonl"
print("Ответ LLM:", llm_answer)
log_query(query ,llm_answer,"logs/manual_query_log.json")
log_rag_version(rag_file_path)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Ответ LLM: Автор имеет в виду иллюзию качества как концепцию, когда продукт или сервис кажется высококачественным и ценными, но на самом деле является низкокачественным и нерезультативным. Это как проекции красоты и совершенства на нечто, которое на самом деле не имеет значения или не приносит реальных результатов. 

В данном случае, автор выделяет случаи, когда дизайн и визуальные элементы проекта перевешивают важность функциональности, конверсии или достижения бизнес-целей. Это когда создатели проектов ориентируются на личный вкус и впечатления от внешнего вида, а не на данные о поведении пользователей и реальные результаты. 

Автор призывает считать главным результатом работы подрядчика не только вовремя сдать макеты, но и добиться бизнес-результатов после запуска.
Запрос записан в лог: logs/munual_query_log.json
✅ Версия RAG-файла записана: version_log.json
